# Transformers, what can they do?

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


# Working with pipelines

Transformers 库中最基础的对象是 `pipeline()` 函数。它将模型与其必要的预处理和后处理步骤连接起来，让我们可以直接输入任何文本并得到清晰易懂的答案：

In [2]:
#HuggingFace Transformers 库内置的一站式工具pipeline，它封装了预处理 + 模型推理 + 后处理全套流程，开箱即用。
from transformers import pipeline

#指定任务：sentiment-analysis = 情感分类；
#自动下载、缓存该任务配套的默认英文预训练模型；
#返回分类器对象classifier。
classifier = pipeline("sentiment-analysis")
#传入待判断文本，模型输出预测标签
classifier("I've been waiting for a HuggingFace course my whole life.")

KeyboardInterrupt: 

默认情况下，此管道会选择一个经过微调以用于英文情感分析的特定预训练模型。当你创建 classifier 对象时，该模型会被下载并缓存。



当你将一些文本传入管道时，主要涉及三个步骤：

1. 文本会被预处理成模型能够理解的格式。


2. 预处理后的输入会被传递给模型。


3. 模型的预测结果经过了后处理，因此你可以理解这些结果。

In [ ]:
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

[{'label': 'POSITIVE', 'score': 0.9598047137260437},
 {'label': 'NEGATIVE', 'score': 0.9994558095932007}]

#Available pipelines for different modalities 适用于不同模态

`pipeline()` 函数支持多种模态，可用于处理文本、图像、音频任务，甚至是多模态任务。

## Zero-shot classification 零样本分类

对未标注的文本进行分类。这是现实项目中的常见场景，因为标注文本通常耗时且需要领域专业知识。针对这种用例，`zero-shot-classification`管道非常强大：它允许你指定分类所用的标签，因此无需依赖预训练模型的标签。你已经了解了模型如何使用这两个标签将句子分类为正面或负面——但它也可以使用你指定的任意其他标签集对文本进行分类。

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445963859558105, 0.111976258456707, 0.043427448719739914]}

##Text generation 文本生成

核心思路是提供一个提示词，模型会通过生成剩余文本自动补全它。这类似于许多手机上的预测文本功能。文本生成包含随机性。

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation")
generator("In this course, we will teach you how to")

[{'generated_text': 'In this course, we will teach you how to understand and use '
                    'data flow and data interchange when handling user data. We '
                    'will be working with one or more of the most commonly used '
                    'data flows — data flows of various types, as seen by the '
                    'HTTP'}]

In [ ]:
from transformers import pipeline

#通过参数 num_return_sequences 控制生成的不同序列数量，
#通过参数 max_length 控制输出文本的总长度。
#通过参数 model从模型库中选择特定模型.

generator = pipeline("text-generation", model="distilgpt2")
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
)

[{'generated_text': 'In this course, we will teach you how to manipulate the world and '
                    'move your mental and physical capabilities to your advantage.'},
 {'generated_text': 'In this course, we will teach you how to become an expert and '
                    'practice realtime, and with a hands on experience on both real '
                    'time and real'}]

##Mask filling 掩码填充

`fill-mask`这个任务的核心是补全给定文本中的空白部分。
此处模型会填入特殊的\<mask>词，该词通常被称为 mask token。

In [ ]:
from transformers import pipeline

unmasker = pipeline("fill-mask")
#top_k 参数控制着你想要显示的可能性数量
unmasker("This course will teach you all about <mask> models.", top_k=2)

[{'sequence': 'This course will teach you all about mathematical models.',
  'score': 0.19619831442832947,
  'token': 30412,
  'token_str': ' mathematical'},
 {'sequence': 'This course will teach you all about computational models.',
  'score': 0.04052725434303284,
  'token': 38163,
  'token_str': ' computational'}]

##Named entity recognition 命名实体识别

命名实体识别（NER）是一项任务，要求模型找出输入文本中哪些部分对应于人物、地点或组织等实体。

模型正确识别出西尔万是一个人（PER）、Hugging Face 是一个组织（ORG）以及布鲁克林是一个地点（LOC）

In [ ]:
from transformers import pipeline

#grouped_entities=True / aggregation_strategy="simple
#告知管道将句子中对应同一实体的部分重新组合在一起：在此示例中，模型正确地将“Hugging”和“Face”合并为一个组织，尽管该名称由多个单词组成。
ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[{'entity_group': 'PER', 'score': 0.99816, 'word': 'Sylvain', 'start': 11, 'end': 18}, 
 {'entity_group': 'ORG', 'score': 0.97960, 'word': 'Hugging Face', 'start': 33, 'end': 45}, 
 {'entity_group': 'LOC', 'score': 0.99321, 'word': 'Brooklyn', 'start': 49, 'end': 57}
]

##uestion answering 问答

`question-answering`管道会利用给定上下文的信息来回答问题.此管道的工作原理是从提供的上下文中提取信息，而非生成答案。

In [ ]:
from transformers import pipeline

question_answerer = pipeline("question-answering")
question_answerer(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)

{'score': 0.6385916471481323, 'start': 33, 'end': 45, 'answer': 'Hugging Face'}

##Summarization 摘要

与文本生成类似，你可以为结果指定max_length或min_length。

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization")
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of
    graduates in traditional engineering disciplines such as mechanical, civil,
    electrical, chemical, and aeronautical engineering declined, but in most of
    the premier American universities engineering curricula now concentrate on
    and encourage largely the study of engineering science. As a result, there
    are declining offerings in engineering subjects dealing with infrastructure,
    the environment, and related issues, and greater concentration on high
    technology subjects, largely supporting increasingly complex scientific
    developments. While the latter is important, it should not be at the expense
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other
    industrial countries in Europe and Asia, continue to encourage and advance
    the teaching of engineering. Both China and India, respectively, graduate
    six and eight times as many traditional engineers as does the United States.
    Other industrial countries at minimum maintain their output, while America
    suffers an increasingly serious decline in the number of engineering graduates
    and a lack of well-educated engineers.
"""
)

[{'summary_text': ' America has changed dramatically during recent years . The '
                  'number of engineering graduates in the U.S. has declined in '
                  'traditional engineering disciplines such as mechanical, civil '
                  ', electrical, chemical, and aeronautical engineering . Rapidly '
                  'developing economies such as China and India, as well as other '
                  'industrial countries in Europe and Asia, continue to encourage '
                  'and advance engineering .'}]

##Translation 翻译

In [ ]:
from transformers import pipeline

translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")
translator("Ce cours est produit par Hugging Face.")

[{'translation_text': 'This course is produced by Hugging Face.'}]

##Image classification 图像分类

In [ ]:
from transformers import pipeline

image_classifier = pipeline(
    task="image-classification", model="google/vit-base-patch16-224"
)
result = image_classifier(
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
)
print(result)

from transformers import pipeline
image_classifier = pipeline(task="image-classification", model="google/vit-base-patch16-224")
result = image_classifier("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg")
print(result)

##Automatic speech recognition 自动语音识别

In [ ]:
from transformers import pipeline

transcriber = pipeline(
    task="automatic-speech-recognition", model="openai/whisper-large-v3"
)
result = transcriber(
    "https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac"
)
print(result)

from transformers import pipeline
transcriber = pipeline(task="automatic-speech-recognition", model="openai/whisper-large-v3")
result = transcriber("https://huggingface.co/datasets/Narsil/asr_dummy/resolve/main/mlk.flac")
print(result)